# SuaraNafas From Scratch — Deteksi TB dari Spektrogram, Tanpa Bobot Pretrained

Notebook ini membangun **model deteksi TB berbasis suara batuk 100% dari nol**
(seperti membuat model sendiri tanpa meminjam bobot apa pun):

- Tanpa ImageNet / ResNet pretrained.
- Tanpa WavLM / Whisper / PaSST.
- Hanya PyTorch (mesin gradien), librosa (DSP spektrogram), dan data CODA-TB.

## Arsitektur

```
klip batuk (~0.55s @16kHz)
        |
  log-mel spektrogram (128 x 36)
        |
  SpectrogramCNN  (Conv/ResBlock, inisialisasi Kaiming - random)
        |  embedding klip 256-d
  Gated Attention-MIL  (agregasi banyak klip per pasien)
        |  representasi pasien 256-d            metadata klinis -> MetaEncoder (32-d)
                        \                         /
                          Late Fusion Concat
                                |
                       Klasifikasi TB+ / TB-
```

## Data
Dataset **CODA-TB** (CODA TB DREAM Challenge, Sage Bionetworks/Synapse `syn31472953`).
Letakkan folder dataset di salah satu path yang didukung pada cell konfigurasi,
atau jalankan mode simulasi untuk menguji pipeline tanpa data.

In [ ]:
# CELL 1: Dependencies
!pip install -q librosa soundfile torch pandas scikit-learn numpy onnx onnxruntime

In [ ]:
# CELL 2: Konfigurasi global
import os
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

SR = 16000
CLIP_DURATION = 0.55
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 256
TARGET_FRAMES = 36
MAX_CLIPS_PER_PATIENT = 24
EMBED_DIM = 256
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# CELL 3: Lokasi dataset CODA-TB
POSSIBLE_DATA_DIRS = [
    "/kaggle/input/datasets/ruchikashirsath/tb-audio",
    "/kaggle/input/tb-audio",
    "./coda_tb_data",
]

def locate_dataset():
    for candidate in POSSIBLE_DATA_DIRS:
        if os.path.exists(candidate):
            return resolve_files(candidate)
    return None

def resolve_files(data_dir):
    clinical_path = additional_path = solicited_path = audio_dir = None
    for root, _, files in os.walk(data_dir):
        wav_count = sum(1 for f in files if f.lower().endswith(".wav"))
        if wav_count > 0 and (audio_dir is None or "solicited" in root.lower()):
            audio_dir = root
        for fname in files:
            lower = fname.lower()
            full = os.path.join(root, fname)
            if lower.endswith(".csv"):
                if "clinical" in lower:
                    clinical_path = full
                elif "additional" in lower:
                    additional_path = full
                elif "solicited" in lower:
                    solicited_path = full
    return {
        "clinical": clinical_path,
        "additional": additional_path,
        "solicited": solicited_path,
        "audio_dir": audio_dir,
    }

DATA_FILES = locate_dataset()
if DATA_FILES is None or any(v is None for v in DATA_FILES.values()):
    print("Dataset CODA-TB tidak ditemukan lengkap. Pipeline akan jalan dalam MODE SIMULASI.")
    DATA_FILES = None
else:
    print("File ditemukan:", DATA_FILES)

In [ ]:
# CELL 4: Preprocessing metadata klinis (skema CODA, identik dengan model pertama)
def preprocess_metadata(clinical_csv_path, additional_csv_path):
    df = pd.merge(
        pd.read_csv(clinical_csv_path),
        pd.read_csv(additional_csv_path),
        on="participant",
        how="inner",
    )

    binary_maps = {
        "sex": {"Male": 1.0, "Female": 0.0},
        "tb_prior": {"Yes": 1.0, "No": 0.0},
        "tb_prior_Pul": {"Yes": 1.0, "No": 0.0},
        "tb_prior_Extrapul": {"Yes": 1.0, "No": 0.0},
        "tb_prior_Unknown": {"Yes": 1.0, "No": 0.0},
        "hemoptysis": {"Yes": 1.0, "No": 0.0},
        "weight_loss": {"Yes": 1.0, "No": 0.0},
        "smoke_lweek": {"Yes": 1.0, "No": 0.0},
        "fever": {"Yes": 1.0, "No": 0.0},
        "night_sweats": {"Yes": 1.0, "No": 0.0},
    }
    for col, mapping in binary_maps.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(0.0)

    df = pd.get_dummies(df, columns=["Country", "HIVstatus"], prefix=["country", "hiv"], dtype=float)

    numeric_cols = ["age", "height", "weight", "reported_cough_dur", "heart_rate", "temperature"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mean())
            df[col] = (df[col] - df[col].mean()) / (df[col].std() + 1e-8)

    exclude_cols = [
        "participant", "type", "tb_status", "Microbiologicreferencestandard",
        "Sputumxpertreferencestandard", "Xpertcombinedsemiquant",
    ]
    feature_cols = [c for c in df.columns if c not in exclude_cols]
    print(f"{len(feature_cols)} fitur klinis: {feature_cols}")
    return df, feature_cols

In [ ]:
# CELL 5: Pipeline log-mel spektrogram (DSP dari nol, tanpa fitur turunan lain)
import librosa

def compute_log_mel(y):
    target_len = int(CLIP_DURATION * SR)
    y = np.asarray(y, dtype=np.float32)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS
    )
    log_mel = librosa.power_to_db(mel, ref=np.max)
    lo, hi = log_mel.min(), log_mel.max()
    norm = (log_mel - lo) / (hi - lo + 1e-8)

    frames = norm.shape[1]
    if frames < TARGET_FRAMES:
        norm = np.pad(norm, ((0, 0), (0, TARGET_FRAMES - frames)))
    else:
        norm = norm[:, :TARGET_FRAMES]
    return norm.astype(np.float32)

In [ ]:
# CELL 6: Dataset multi-instance per pasien (cache RAM)
class MultiInstanceCoughDataset(torch.utils.data.Dataset):
    def __init__(self, metadata_df, feature_cols, solicited_csv_path, audio_dir):
        self.feature_cols = feature_cols
        self.audio_dir = audio_dir
        self._df_lookup = metadata_df.set_index("participant")

        solicited = pd.read_csv(solicited_csv_path)
        valid = set(metadata_df["participant"])
        solicited = solicited[solicited["participant"].isin(valid)]
        audio_map = solicited.groupby("participant")["filename"].apply(list).to_dict()
        self.participants = [p for p in metadata_df["participant"].values if p in audio_map]

        print(f"Menghitung spektrogram untuk {len(self.participants)} pasien ...")
        self.cache = {}
        for idx, participant in enumerate(self.participants):
            specs = []
            for fname in audio_map[participant][:MAX_CLIPS_PER_PATIENT]:
                try:
                    y, _ = librosa.load(os.path.join(audio_dir, fname), sr=SR, duration=CLIP_DURATION)
                except Exception:
                    continue
                specs.append(torch.tensor(compute_log_mel(y)).unsqueeze(0))
            if len(specs) == 0:
                continue
            self.cache[participant] = torch.stack(specs)
            if idx % 200 == 0:
                print(f"  {idx}/{len(self.participants)}")
        self.participants = [p for p in self.participants if p in self.cache]
        print(f"Selesai. Pasien valid: {len(self.participants)}")

    def __len__(self):
        return len(self.participants)

    def __getitem__(self, idx):
        participant = self.participants[idx]
        row = self.metadata_row(participant)
        meta = torch.tensor(row[self.feature_cols].values.astype(np.float32))
        label = torch.tensor(int(row["tb_status"]), dtype=torch.long)
        return self.cache[participant], meta, label

    def metadata_row(self, participant):
        return self._df_lookup.loc[participant]

def collate_patients(batch):
    return (
        [item[0] for item in batch],
        torch.stack([item[1] for item in batch]),
        torch.stack([item[2] for item in batch]),
    )

In [ ]:
# CELL 7: CNN spektrogram DARI NOL (inisialisasi acak, tanpa pretrained)
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(ConvBlock(channels, channels), ConvBlock(channels, channels))

    def forward(self, x):
        return x + self.block(x)

def kaiming_init(module):
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.ones_(module.weight)
        nn.init.zeros_(module.bias)

class SpectrogramCNN(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            nn.MaxPool2d(2),
            ConvBlock(32, 64),
            nn.MaxPool2d(2),
            ResidualBlock(64),
            ConvBlock(64, 128),
            nn.MaxPool2d(2),
            ResidualBlock(128),
            nn.AdaptiveAvgPool2d(1),
        )
        self.projector = nn.Linear(128, embed_dim)
        self.apply(kaiming_init)

    def forward(self, x):
        pooled = self.features(x).flatten(1)
        return self.projector(pooled)

class GatedAttentionMIL(nn.Module):
    def __init__(self, feature_dim=EMBED_DIM, hidden_dim=128):
        super().__init__()
        self.attention_v = nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.Tanh())
        self.attention_u = nn.Sequential(nn.Linear(feature_dim, hidden_dim), nn.Sigmoid())
        self.attention_w = nn.Linear(hidden_dim, 1)

    def forward(self, embeddings, mask=None):
        scores = self.attention_w(self.attention_v(embeddings) * self.attention_u(embeddings)).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(~mask.bool(), float("-inf"))
        weights = F.softmax(scores, dim=0)
        return weights.unsqueeze(-1) * embeddings

In [ ]:
# CELL 8: Model multimodal gabungan (masih dari nol)
class MetaEncoder(nn.Module):
    def __init__(self, meta_dim, out_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(meta_dim),
            nn.Linear(meta_dim, out_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(out_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)

class SuaraNafasScratchModel(nn.Module):
    def __init__(self, meta_dim, embed_dim=EMBED_DIM):
        super().__init__()
        self.cnn = SpectrogramCNN(embed_dim)
        self.attention = GatedAttentionMIL(embed_dim)
        self.meta_encoder = MetaEncoder(meta_dim)
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 2),
        )

    def encode_patient(self, clips):
        embeddings = self.cnn(clips)
        weighted = self.attention(embeddings)
        return weighted.sum(dim=0)

    def forward(self, patient_specs, metadata):
        patient_embeddings = torch.stack([self.encode_patient(specs.to(device)) for specs in patient_specs])
        fused = torch.cat((patient_embeddings, self.meta_encoder(metadata.to(device))), dim=1)
        return self.classifier(fused)

In [ ]:
# CELL 9: SpecAugment manual (masking waktu & frekuensi)
class SpecAugment(nn.Module):
    def __init__(self, freq_mask_max=15, time_mask_max=15):
        super().__init__()
        self.freq_mask_max = freq_mask_max
        self.time_mask_max = time_mask_max

    def forward(self, x):
        if not self.training:
            return x
        x = x.clone()
        _, _, height, width = x.shape
        for i in range(x.shape[0]):
            f = int(torch.randint(0, self.freq_mask_max + 1, (1,)).item())
            if f > 0:
                f0 = int(torch.randint(0, height - f + 1, (1,)).item())
                x[i, :, f0:f0 + f, :] = 0.0
            t = int(torch.randint(0, self.time_mask_max + 1, (1,)).item())
            if t > 0:
                t0 = int(torch.randint(0, width - t + 1, (1,)).item())
                x[i, :, :, t0:t0 + t] = 0.0
        return x

In [ ]:
# CELL 10: Split stratified per pasien + bobot kelas + loader
from sklearn.model_selection import StratifiedShuffleSplit

def build_loaders(dataset, batch_size):
    labels = [dataset[i][2].item() for i in range(len(dataset))]
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(splitter.split(range(len(dataset)), labels))

    train_ds = torch.utils.data.Subset(dataset, train_idx)
    val_ds = torch.utils.data.Subset(dataset, val_idx)

    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        collate_fn=collate_patients, drop_last=len(train_ds) >= batch_size,
    )
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_patients,
    )
    return train_loader, val_loader

def compute_class_weights(labels):
    counts = np.bincount(np.asarray(labels, dtype=np.int64), minlength=2).astype(np.float64)
    weights = np.sqrt(counts.sum() / np.maximum(counts, 1.0))
    weights = weights / weights.sum() * 2.0
    return torch.tensor(weights, dtype=torch.float32, device=device)

In [ ]:
# CELL 11: Training & evaluasi (AUROC, threshold Youden J, balanced accuracy)
from sklearn.metrics import roc_auc_score, roc_curve, balanced_accuracy_score

def evaluate(model, loader):
    model.eval()
    all_probs, all_labels, total_loss = [], [], 0.0
    with torch.no_grad():
        for specs, meta, labels in loader:
            logits = model(specs, meta)
            total_loss += F.cross_entropy(logits, labels.to(device)).item()
            probs = F.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    metrics = {"loss": total_loss / max(len(loader), 1)}
    if len(set(all_labels)) > 1:
        metrics["auroc"] = roc_auc_score(all_labels, all_probs)
        fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
        best = int(np.argmax(tpr - fpr))
        metrics["threshold"] = thresholds[best]
        predictions = (np.asarray(all_probs) >= metrics["threshold"]).astype(int)
        metrics["balanced_acc"] = balanced_accuracy_score(all_labels, predictions)
    return metrics

def train_model(model, train_loader, val_loader, class_weights, epochs=40, lr=3e-4):
    criterion_weights = class_weights
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=3)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs - 3, 1), eta_min=1e-5)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[3])
    spec_augment = SpecAugment()

    best_auroc, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        running = 0.0
        for specs, meta, labels in train_loader:
            if len(specs) == 1:
                continue
            optimizer.zero_grad()
            augmented = [spec_augment(s.to(device)) for s in specs]
            logits = model(augmented, meta)
            loss = F.cross_entropy(logits, labels.to(device), weight=criterion_weights, label_smoothing=0.05)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running += loss.item()

        val_metrics = evaluate(model, val_loader)
        scheduler.step()

        marker = ""
        if val_metrics.get("auroc", 0.0) > best_auroc:
            best_auroc = val_metrics.get("auroc", 0.0)
            best_state = copy.deepcopy(model.state_dict())
            marker = " <- terbaik"

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"train {running / max(len(train_loader), 1):.4f} | "
            f"val {val_metrics['loss']:.4f} | "
            f"AUROC {val_metrics.get('auroc', float('nan')):.4f} | "
            f"bAcc {val_metrics.get('balanced_acc', float('nan')):.4f}{marker}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Bobot terbaik dipulihkan (AUROC {best_auroc:.4f})")
    return model

In [ ]:
# CELL 12: Jalankan - simulasi bila tanpa data, training nyata bila dataset ada
def run_simulation():
    print("MODE SIMULASI: memverifikasi arsitektur dengan data acak.")
    meta_dim = 18
    model = SuaraNafasScratchModel(meta_dim=meta_dim).to(device)
    specs_a = torch.randn(5, 1, N_MELS, TARGET_FRAMES).to(device)
    specs_b = torch.randn(2, 1, N_MELS, TARGET_FRAMES).to(device)
    meta = torch.randn(2, meta_dim).to(device)
    labels = torch.tensor([1, 0], device=device)

    logits = model([specs_a, specs_b], meta)
    print(f"Bentuk logits: {tuple(logits.shape)}")

    loss = F.cross_entropy(logits, labels)
    loss.backward()
    print(f"Backward OK, loss simulasi: {loss.item():.4f}")

if DATA_FILES is None:
    run_simulation()
else:
    df_meta, feature_cols = preprocess_metadata(DATA_FILES["clinical"], DATA_FILES["additional"])
    dataset = MultiInstanceCoughDataset(df_meta, feature_cols, DATA_FILES["solicited"], DATA_FILES["audio_dir"])

    if len(dataset) == 0:
        print("Tidak ada pasien dengan audio. Jalankan ulang setelah dataset siap.")
    else:
        labels_all = [dataset[i][2].item() for i in range(len(dataset))]
        pos_rate = np.mean(labels_all)
        print(f"Pasien: {len(dataset)} | proporsi TB+: {pos_rate * 100:.1f}%")

        train_loader, val_loader = build_loaders(dataset, batch_size=8)
        model = SuaraNafasScratchModel(meta_dim=len(feature_cols)).to(device)
        class_weights = compute_class_weights(labels_all)
        model = train_model(model, train_loader, val_loader, class_weights, epochs=40)
        torch.save(model.state_dict(), "suaranafas_scratch.pt")
        print("Bobot disimpan ke suaranafas_scratch.pt")

In [ ]:
# CELL 13: Ekspor ONNX untuk backend FastAPI
class PaddedInferenceWrapper(nn.Module):
    def __init__(self, model, max_clips=8):
        super().__init__()
        self.model = model
        self.max_clips = max_clips

    def forward(self, clips, clip_mask, metadata):
        batch_size, num_clips = clips.shape[0], clips.shape[1]
        embeddings = self.model.cnn(clips.flatten(0, 1)).view(batch_size, num_clips, -1)
        scores = self.model.attention.attention_w(
            self.model.attention.attention_v(embeddings) * self.model.attention.attention_u(embeddings)
        ).squeeze(-1)
        neg_inf = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(~clip_mask.bool(), neg_inf)
        weights = F.softmax(scores, dim=1).unsqueeze(-1)
        patient_embedding = (weights * embeddings).sum(dim=1)
        fused = torch.cat((patient_embedding, self.model.meta_encoder(metadata)), dim=1)
        return self.model.classifier(fused)

def export_onnx(model, meta_dim, path="suaranafas_scratch.onnx", max_clips=8):
    model = model.cpu().eval()
    wrapper = PaddedInferenceWrapper(model, max_clips).cpu().eval()
    demo_clips = torch.randn(1, max_clips, 1, N_MELS, TARGET_FRAMES)
    demo_mask = torch.ones(1, max_clips)
    demo_meta = torch.randn(1, meta_dim)
    torch.onnx.export(
        wrapper,
        (demo_clips, demo_mask, demo_meta),
        path,
        input_names=["clips", "clip_mask", "metadata"],
        output_names=["tb_logits"],
        dynamic_axes={
            "clips": {0: "batch"},
            "clip_mask": {0: "batch"},
            "metadata": {0: "batch"},
            "tb_logits": {0: "batch"},
        },
        opset_version=17,
    )
    print(f"Ekspor ONNX selesai: {path}")

    import onnxruntime as ort
    session = ort.InferenceSession(path)
    outputs = session.run(
        None,
        {
            "clips": demo_clips.numpy(),
            "clip_mask": demo_mask.numpy(),
            "metadata": demo_meta.numpy(),
        },
    )
    print(f"Smoke test onnxruntime OK, bentuk keluaran: {outputs[0].shape}")

if DATA_FILES is None:
    sim_model = SuaraNafasScratchModel(meta_dim=18).to(device).eval()
    export_onnx(sim_model, meta_dim=18)
else:
    export_onnx(model.cpu(), meta_dim=len(feature_cols))

## Integrasi ke Backend

Hasil `suaranafas_scratch.onnx` siap dipakai FastAPI endpoint `/predict`:

| Input wrapper | Bentuk | Keterangan |
|---|---|---|
| `clips` | `(batch, <=8, 1, 128, 36)` | log-mel spektrogram tiap klip batuk |
| `clip_mask` | `(batch, <=8)` | 1 = klip valid, 0 = padding |
| `metadata` | `(batch, n_fitur_klinis)` | urutan kolom = `feature_cols` dari Cell 4 |

Metadata JSON dari aplikasi web (`sex`, `age`, `height`, `weight`, `reported_cough_dur`,
gejala yes/no, `Country`, `HIVstatus`) di-encode memakai skema yang sama dengan
`preprocess_metadata` sehingga frontend, notebook lama, dan notebook ini kompatibel.

> Catatan performa: model dari nol butuh lebih banyak epoch/data dibanding pendekatan
> pretrained. Bandingkan AUROC-nya dengan notebook `main-model-experiments.ipynb`
> (ResNet18 pretrained) untuk laporan hackathon.